# fold_frontier_failure_attribution_v1

Purpose: inspect all existing fold-level model-selection replay evidence after `return_preserving_hybrid_target_bc_replay_v1` failed, then identify the smallest remaining blocker before another training run.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `return_preserving_hybrid_target_bc_replay_v1`; reference `soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05`.
- Role: diagnose. This card does not train, retain, or promote an actor.
- Target window: required walk-forward windows `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Accepted trade-off: MaxDD <20%; `wf_2020` return >= risk-cap floor 69.60%; `wf_2022` and `wf_2023_2024` relative return >= +30pp.
- Changed layer: `model_selection`, `evidence_routing`, `target_repair_design`.
- Hypothesis: the broad hybrid target failed because it damaged folds that already had passing candidates; the next useful step is a fold-aware failure attribution that isolates the unresolved `wf_2020` timing gap.
- Acceptance criteria: produce a normalized fold frontier, identify whether any existing candidate can satisfy all folds, and write the next target-repair mechanism without allowing training if the frontier is incomplete.
- Closest historical success: fold-level passing candidates already exist for `wf_2022` and `wf_2023_2024` in checkpoint metric/drift replay cards.
- Closest historical failure: every available `wf_2020` candidate still misses the calibrated risk-cap return floor or breaches MaxDD.
- Why not duplicate: this card aggregates existing candidate rows across cards and diagnoses the remaining fold-specific blocker instead of running another target/BC screen.


In [1]:

from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT = 'fold_frontier_failure_attribution_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
MODEL_SELECTION_ROOT = ROOT / 'runtime/model_selection/card'
GATE_CONTRACT_PATH = MODEL_SELECTION_ROOT / 'wf2020_reference_floor_gate_calibration_v1/training_gate_contract.json'
DIRECT_SUMMARY_PATH = MODEL_SELECTION_ROOT / 'return_preserving_hybrid_target_bc_replay_v1/summary.json'
TARGET_MAX_DD = 20.0
REQUIRED_FOLDS = ['wf_2020', 'wf_2022', 'wf_2023_2024']


def _number(value: Any) -> float | None:
    try:
        if value is None or value == '':
            return None
        out = float(value)
        return out if np.isfinite(out) else None
    except (TypeError, ValueError):
        return None


def _bool(value: Any) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {'1', 'true', 'yes', 'y'}
    return bool(value)


def to_json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(to_json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


gate_contract = json.loads(GATE_CONTRACT_PATH.read_text(encoding='utf-8'))
direct_summary = json.loads(DIRECT_SUMMARY_PATH.read_text(encoding='utf-8'))
target_relative = {str(k): float(v) for k, v in gate_contract.get('target_relative_return_by_fold_pp', {}).items()}
wf2020_return_floor = float(gate_contract['wf2020_reference_floor_return_pct'])
wf2020_floor_max_dd = float(gate_contract['wf2020_reference_floor_max_dd_pct'])


In [2]:

rows: list[dict[str, Any]] = []

for path in sorted(MODEL_SELECTION_ROOT.glob('*/checkpoint_candidate_metrics.csv')):
    experiment = path.parent.name
    frame = pd.read_csv(path)
    for row in frame.to_dict('records'):
        fold = str(row.get('fold') or '')
        if fold not in REQUIRED_FOLDS:
            continue
        rows.append({
            'source_card': experiment,
            'source_file': str(path.relative_to(ROOT)),
            'fold': fold,
            'candidate': str(row.get('candidate') or ''),
            'candidate_kind': str(row.get('candidate_kind') or ''),
            'iter': _number(row.get('iter')),
            'return_pct': _number(row.get('test_return')),
            'relative_return_pp': _number(row.get('test_relative_return')),
            'max_dd_pct': _number(row.get('test_max_dd')),
            'avg_pos': _number(row.get('test_avg_pos')),
            'candidate_gate_source': _bool(row.get('candidate_gate')),
            'target_gate_source': _bool(row.get('target_gate')),
            'drift_gate_source': _bool(row.get('drift_gate')),
            'valid_return_pct': _number(row.get('valid_return')),
            'valid_max_dd_pct': _number(row.get('valid_max_dd')),
            'selection_loss': _number(row.get('selection_loss')),
            'daily_path': '',
        })

for path in sorted(MODEL_SELECTION_ROOT.glob('*/fold*_replay_metrics.csv')):
    experiment = path.parent.name
    frame = pd.read_csv(path)
    for row in frame.to_dict('records'):
        fold = str(row.get('fold') or '')
        if fold not in REQUIRED_FOLDS:
            continue
        rows.append({
            'source_card': experiment,
            'source_file': str(path.relative_to(ROOT)),
            'fold': fold,
            'candidate': str(row.get('candidate') or row.get('selected_candidate') or experiment),
            'candidate_kind': str(row.get('variant') or row.get('selected_kind') or 'official_replay'),
            'iter': _number(row.get('selected_iter')),
            'return_pct': _number(row.get('return') if 'return' in row else row.get('selected_return')),
            'relative_return_pp': _number(row.get('relative_return') if 'relative_return' in row else row.get('selected_relative_return')),
            'max_dd_pct': _number(row.get('max_dd') if 'max_dd' in row else row.get('selected_max_dd')),
            'avg_pos': _number(row.get('avg_pos') if 'avg_pos' in row else row.get('selected_avg_pos')),
            'candidate_gate_source': _bool(row.get('fold_gate') if 'fold_gate' in row else row.get('selected_candidate_gate')),
            'target_gate_source': _bool(row.get('return_floor_gate') if fold == 'wf_2020' and 'return_floor_gate' in row else row.get('relative_gate')),
            'drift_gate_source': _bool(row.get('max_dd_gate')),
            'valid_return_pct': None,
            'valid_max_dd_pct': None,
            'selection_loss': None,
            'daily_path': str(row.get('daily_path') or ''),
        })

frontier = pd.DataFrame(rows).drop_duplicates(
    subset=['source_card', 'source_file', 'fold', 'candidate', 'return_pct', 'relative_return_pp', 'max_dd_pct'],
    keep='first',
).reset_index(drop=True)

if frontier.empty:
    raise RuntimeError('no fold candidate evidence found')

frontier['target_relative_return_pp'] = frontier['fold'].map(target_relative).fillna(30.0)
frontier['target_max_dd_pct'] = TARGET_MAX_DD
frontier['wf2020_reference_floor_return_pct'] = np.where(frontier['fold'].eq('wf_2020'), wf2020_return_floor, np.nan)
frontier['max_dd_gate'] = frontier['max_dd_pct'].lt(TARGET_MAX_DD)
frontier['relative_gate'] = frontier['relative_return_pp'].ge(frontier['target_relative_return_pp'])
frontier['wf2020_return_floor_gate'] = np.where(
    frontier['fold'].eq('wf_2020'),
    frontier['return_pct'].ge(wf2020_return_floor),
    True,
)
frontier['fold_gate'] = frontier['max_dd_gate'] & frontier['relative_gate'] & frontier['wf2020_return_floor_gate']
frontier['max_dd_gap_pct'] = np.maximum(0.0, frontier['max_dd_pct'] - TARGET_MAX_DD)
frontier['relative_gap_pp'] = np.maximum(0.0, frontier['target_relative_return_pp'] - frontier['relative_return_pp'])
frontier['wf2020_return_gap_pp'] = np.where(
    frontier['fold'].eq('wf_2020'),
    np.maximum(0.0, wf2020_return_floor - frontier['return_pct']),
    0.0,
)
frontier['hard_gap_score'] = frontier['max_dd_gap_pct'] * 100.0 + frontier['relative_gap_pp'] + frontier['wf2020_return_gap_pp']
frontier.to_csv(CARD / 'normalized_candidate_frontier.csv', index=False)

frontier_summary = frontier.groupby('fold').agg(
    candidates=('candidate', 'count'),
    passing_candidates=('fold_gate', 'sum'),
    best_return_pct=('return_pct', 'max'),
    best_relative_return_pp=('relative_return_pp', 'max'),
    lowest_max_dd_pct=('max_dd_pct', 'min'),
).reset_index()
frontier_summary.to_csv(CARD / 'fold_candidate_summary.csv', index=False)
display(frontier_summary)


,fold,candidates,passing_candidates,best_return_pct,best_relative_return_pp,lowest_max_dd_pct
0,wf_2020,20,0,61.324050,-43.627240,14.831863
1,wf_2022,24,15,-6.273652,75.880351,10.123049
2,wf_2023_2024,24,12,17.006850,53.970997,8.955769


In [3]:

def rank_fold(group: pd.DataFrame) -> pd.DataFrame:
    work = group.copy()
    if group.name == 'wf_2020':
        # First: exact gate. If none, prefer MaxDD-safe candidates closest to the calibrated return floor.
        work = work.sort_values(
            ['fold_gate', 'max_dd_gate', 'wf2020_return_gap_pp', 'relative_gap_pp', 'max_dd_gap_pct', 'return_pct'],
            ascending=[False, False, True, True, True, False],
        )
    else:
        work = work.sort_values(
            ['fold_gate', 'max_dd_gate', 'relative_gap_pp', 'max_dd_gap_pct', 'return_pct'],
            ascending=[False, False, True, True, False],
        )
    work['fold_rank'] = range(1, len(work) + 1)
    return work

ranked = frontier.groupby('fold', group_keys=False).apply(rank_fold).reset_index(drop=True)
best_by_fold = ranked[ranked['fold_rank'].eq(1)].copy()

# Pareto frontier per fold: maximize return, minimize MaxDD.
pareto_rows = []
for fold, group in frontier.groupby('fold'):
    valid = group.dropna(subset=['return_pct', 'max_dd_pct']).copy()
    for _, row in valid.iterrows():
        dominated = valid[
            (valid['return_pct'] >= row['return_pct'])
            & (valid['max_dd_pct'] <= row['max_dd_pct'])
            & ((valid['return_pct'] > row['return_pct']) | (valid['max_dd_pct'] < row['max_dd_pct']))
        ]
        if dominated.empty:
            pareto_rows.append(row.to_dict())
pareto = pd.DataFrame(pareto_rows).sort_values(['fold', 'return_pct'], ascending=[True, False])

near_miss = ranked[
    ranked['fold'].eq('wf_2020')
    & ranked['max_dd_pct'].lt(TARGET_MAX_DD + 2.0)
    & ranked['return_pct'].notna()
].sort_values(['wf2020_return_gap_pp', 'max_dd_gap_pct', 'return_pct'], ascending=[True, True, False])

ranked.to_csv(CARD / 'ranked_candidate_frontier.csv', index=False)
best_by_fold.to_csv(CARD / 'best_existing_by_fold.csv', index=False)
pareto.to_csv(CARD / 'fold_pareto_frontier.csv', index=False)
near_miss.to_csv(CARD / 'wf2020_near_miss.csv', index=False)

display(best_by_fold[['fold', 'source_card', 'candidate', 'return_pct', 'relative_return_pp', 'max_dd_pct', 'avg_pos', 'fold_gate', 'wf2020_return_gap_pp', 'max_dd_gap_pct']])
display(near_miss[['source_card', 'candidate', 'return_pct', 'relative_return_pp', 'max_dd_pct', 'avg_pos', 'fold_gate', 'wf2020_return_gap_pp', 'max_dd_gap_pct']].head(12))


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_22475/4294833180.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ranked = frontier.groupby('fold', group_keys=False).apply(rank_fold).reset_index(drop=True)


,fold,source_card,candidate,return_pct,relative_return_pp,max_dd_pct,avg_pos,fold_gate,wf2020_return_gap_pp,max_dd_gap_pct
0,wf_2020,bull_reentry_three_fold_checkpoint_metric_repl...,bc_initialization,55.597960,-277.735388,15.489488,0.406077,False,14.006592,0.0
20,wf_2022,cap10_three_fold_checkpoint_drift_selector_v1,bc_initialization,-6.273652,47.131485,18.657201,0.252115,True,0.000000,0.0
44,wf_2023_2024,cap10_three_fold_checkpoint_drift_selector_v1,ppo_retained_iter_4,1.282379,53.970997,10.940667,0.219862,True,0.000000,0.0


,source_card,candidate,return_pct,relative_return_pp,max_dd_pct,avg_pos,fold_gate,wf2020_return_gap_pp,max_dd_gap_pct
12,cap10_three_fold_checkpoint_metric_replay_v2,ppo_retained_iter_4,61.324050,-272.009298,20.918285,0.358682,False,8.280502,0.918285
13,cap10_three_fold_checkpoint_metric_replay_v2,ppo_selected_best,61.324050,-272.009298,20.918285,0.358682,False,8.280502,0.918285
0,bull_reentry_three_fold_checkpoint_metric_repl...,bc_initialization,55.597960,-277.735388,15.489488,0.406077,False,14.006592,0.000000
1,bull_reentry_three_fold_checkpoint_metric_repl...,ppo_retained_iter_0_bc_equivalent,55.597960,-277.735388,15.489488,0.406077,False,14.006592,0.000000
2,bull_reentry_three_fold_checkpoint_metric_repl...,ppo_retained_iter_4,50.224828,-283.108519,15.475617,0.270306,False,19.379723,0.000000
3,bull_reentry_three_fold_checkpoint_metric_repl...,ppo_selected_best,50.224828,-283.108519,15.475617,0.270306,False,19.379723,0.000000
4,cap10_three_fold_checkpoint_drift_selector_v1,bc_initialization,46.810017,-285.449375,19.417943,0.366624,False,22.794534,0.000000
5,cap10_three_fold_checkpoint_drift_selector_v1,ppo_retained_iter_0_bc_equivalent,46.810017,-285.449375,19.417943,0.366624,False,22.794534,0.000000
6,cap10_three_fold_checkpoint_metric_replay_v2,bc_initialization,40.580326,-292.753021,14.984701,0.405426,False,29.024226,0.000000
7,cap10_three_fold_checkpoint_metric_replay_v2,ppo_retained_iter_0_bc_equivalent,40.580326,-292.753021,14.984701,0.405426,False,29.024226,0.000000


In [4]:

fold_records = best_by_fold.to_dict('records')
selected_by_fold = {}
for row in fold_records:
    fold = str(row['fold'])
    selected_by_fold[fold] = {
        'fold': fold,
        'status': 'evaluated',
        'selected_candidate': f"{row['source_card']}:{row['candidate']}",
        'selected_kind': str(row.get('candidate_kind') or 'candidate_frontier'),
        'selected_iter': row.get('iter'),
        'selected_return': row.get('return_pct'),
        'selected_relative_return': row.get('relative_return_pp'),
        'selected_max_dd': row.get('max_dd_pct'),
        'selected_avg_pos': row.get('avg_pos'),
        'selected_target_gate': bool(row.get('relative_gate')) and (fold != 'wf_2020' or bool(row.get('wf2020_return_floor_gate'))),
        'selected_candidate_gate': bool(row.get('fold_gate')),
        'selected_drift_gate': bool(row.get('max_dd_gate')),
        'target_relative_return': row.get('target_relative_return_pp'),
        'target_max_dd': TARGET_MAX_DD,
        'wf2020_return_floor_gap_pp': row.get('wf2020_return_gap_pp') if fold == 'wf_2020' else None,
        'max_dd_gap_pct': row.get('max_dd_gap_pct'),
        'relative_gap_pp': row.get('relative_gap_pp'),
        'source_file': row.get('source_file'),
        'daily_path': row.get('daily_path'),
    }

missing_folds = [fold for fold in REQUIRED_FOLDS if fold not in selected_by_fold]
failed_folds = [fold for fold, row in selected_by_fold.items() if not row['selected_candidate_gate']]
fold_gate_pass_count = sum(1 for row in selected_by_fold.values() if row['selected_candidate_gate'])
max_dd_pass_count = sum(1 for row in selected_by_fold.values() if row['selected_drift_gate'])
relative_pass_count = sum(1 for row in selected_by_fold.values() if (row['selected_relative_return'] or -1e9) >= row['target_relative_return'])
return_floor_pass_count = sum(1 for fold, row in selected_by_fold.items() if fold != 'wf_2020' or (row['selected_return'] or -1e9) >= wf2020_return_floor)

wf2020_best_safe = best_by_fold[best_by_fold['fold'].eq('wf_2020')].iloc[0].to_dict()
wf2020_best_return = frontier[frontier['fold'].eq('wf_2020')].sort_values(['return_pct'], ascending=False).iloc[0].to_dict()
wf2020_best_near_miss = near_miss.iloc[0].to_dict() if not near_miss.empty else wf2020_best_safe

if fold_gate_pass_count == len(REQUIRED_FOLDS):
    decision = 'fold_frontier_all_existing_candidates_pass_no_promotion'
    next_mechanism = 'run_fixed_split_ytd_retention_training_and_actor_review_before_promotion'
else:
    decision = 'fold_frontier_wf2020_timing_gap_blocks_retention'
    next_mechanism = 'build_wf2020_near_miss_return_timing_repair_then_rerun_fold_aware_three_fold_replay'

repair_plan = {
    'decision': decision,
    'next_mechanism': next_mechanism,
    'wf2020_primary_gap': {
        'best_maxdd_safe_candidate': f"{wf2020_best_safe['source_card']}:{wf2020_best_safe['candidate']}",
        'best_maxdd_safe_return_pct': wf2020_best_safe.get('return_pct'),
        'best_maxdd_safe_max_dd_pct': wf2020_best_safe.get('max_dd_pct'),
        'best_maxdd_safe_return_gap_pp': wf2020_best_safe.get('wf2020_return_gap_pp'),
        'highest_return_candidate': f"{wf2020_best_return['source_card']}:{wf2020_best_return['candidate']}",
        'highest_return_pct': wf2020_best_return.get('return_pct'),
        'highest_return_max_dd_pct': wf2020_best_return.get('max_dd_pct'),
        'highest_return_max_dd_gap_pct': wf2020_best_return.get('max_dd_gap_pct'),
        'near_miss_candidate': f"{wf2020_best_near_miss['source_card']}:{wf2020_best_near_miss['candidate']}",
        'near_miss_return_pct': wf2020_best_near_miss.get('return_pct'),
        'near_miss_max_dd_pct': wf2020_best_near_miss.get('max_dd_pct'),
        'near_miss_return_gap_pp': wf2020_best_near_miss.get('wf2020_return_gap_pp'),
        'near_miss_max_dd_gap_pct': wf2020_best_near_miss.get('max_dd_gap_pct'),
    },
    'repair_constraints': [
        'Do not restart from full risk-cap replacement; it underparticipated wf_2020 and damaged 2023-2024 relative return.',
        'Start from the bull-reentry BC/near-miss wf_2020 surface because it is within 14.01pp return of the risk-cap floor while keeping MaxDD under 20%.',
        'Use the 61.32% return / 20.92% MaxDD wf_2020 near-miss as the upper participation edge and repair its sub-1pp drawdown breach, rather than adding broad exposure.',
        'Preserve already passing wf_2022 and wf_2023_2024 candidates as comparison baselines; any new target must not regress those folds.',
    ],
}
write_json(CARD / 'repair_plan.json', repair_plan)

summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'return_preserving_hybrid_target_bc_replay_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
    'changed_layer': ['model_selection', 'evidence_routing', 'target_repair_design'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'next_training_allowed_after_replay': False,
    'next_mechanism': next_mechanism,
    'required_wf_windows': REQUIRED_FOLDS,
    'target_max_dd_pct': TARGET_MAX_DD,
    'target_relative_return_by_fold_pp': target_relative,
    'wf2020_reference_floor_return_pct': wf2020_return_floor,
    'wf2020_reference_floor_max_dd_pct': wf2020_floor_max_dd,
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'fold_gate_pass_count': fold_gate_pass_count,
    'max_dd_pass_count': max_dd_pass_count,
    'relative_pass_count': relative_pass_count,
    'return_floor_pass_count': return_floor_pass_count,
    'selected_by_fold': selected_by_fold,
    'wf2020_primary_gap': repair_plan['wf2020_primary_gap'],
    'source_summary': {
        'direct_decision': direct_summary.get('decision'),
        'direct_selected_candidate': direct_summary.get('selected_candidate'),
        'direct_fold_gate_pass_count': direct_summary.get('fold_gate_pass_count'),
        'candidate_rows_scanned': int(len(frontier)),
        'cards_scanned': int(frontier['source_card'].nunique()),
    },
    'source_decisions': {
        'return_preserving_hybrid_target_bc_replay_v1': direct_summary.get('decision'),
        'wf2020_reference_floor_gate_calibration_v1': 'wf2020_reference_floor_gate_calibrated_no_promotion',
        'bull_reentry_three_fold_checkpoint_metric_replay_v1': 'three_fold_checkpoint_metric_replay_partial_pass_no_promotion',
    },
    'sidecars': [
        'normalized_candidate_frontier.csv',
        'fold_candidate_summary.csv',
        'ranked_candidate_frontier.csv',
        'best_existing_by_fold.csv',
        'fold_pareto_frontier.csv',
        'wf2020_near_miss.csv',
        'repair_plan.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
    'conclusion': [
        f"Existing candidates pass {fold_gate_pass_count}/{len(REQUIRED_FOLDS)} folds as a combined frontier; no full retention training is allowed from this card.",
        f"wf_2020 best MaxDD-safe candidate is {repair_plan['wf2020_primary_gap']['best_maxdd_safe_candidate']} at {repair_plan['wf2020_primary_gap']['best_maxdd_safe_return_pct']:.2f}% return and {repair_plan['wf2020_primary_gap']['best_maxdd_safe_max_dd_pct']:.2f}% MaxDD, leaving {repair_plan['wf2020_primary_gap']['best_maxdd_safe_return_gap_pp']:.2f}pp to the 69.60% floor.",
        f"wf_2020 highest-return near-miss reaches {repair_plan['wf2020_primary_gap']['near_miss_return_pct']:.2f}% return with {repair_plan['wf2020_primary_gap']['near_miss_max_dd_pct']:.2f}% MaxDD, so the next repair should target a sub-1pp drawdown breach instead of replacing the whole target with risk-cap labels.",
        'wf_2022 and wf_2023_2024 already have passing candidates in existing checkpoint replay evidence; they should be protected comparison folds in the next run.',
    ],
}
selection = {
    **summary,
    'selected_by_fold': selected_by_fold,
    'repair_plan': repair_plan,
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'Decision: `{decision}`.',
    '',
    f'Fold gates passed by existing frontier: `{fold_gate_pass_count}/{len(REQUIRED_FOLDS)}`. MaxDD gates: `{max_dd_pass_count}/{len(REQUIRED_FOLDS)}`. Relative gates: `{relative_pass_count}/{len(REQUIRED_FOLDS)}`. Return-floor gates: `{return_floor_pass_count}/{len(REQUIRED_FOLDS)}`.',
    '',
    *[f'- {line}' for line in summary['conclusion']],
    '',
    f'Next action: `{next_mechanism}`.',
    '',
    'Promotion note: `promotion_allowed=false`, `actor_retention_allowed=false`; this card diagnoses the fold frontier and target-repair boundary only.',
]
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines) + '\n', encoding='utf-8')
summary


{'experiment': 'fold_frontier_failure_attribution_v1',
 'created_at': '2026-06-14T01:50:43.465132+00:00',
 'decision': 'fold_frontier_wf2020_timing_gap_blocks_retention',
 'role': 'diagnose',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'return_preserving_hybrid_target_bc_replay_v1',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
 'changed_layer': ['model_selection',
  'evidence_routing',
  'target_repair_design'],
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'next_training_allowed_after_replay': False,
 'next_mechanism': 'build_wf2020_near_miss_return_timing_repair_then_rerun_fold_aware_three_fold_replay',
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'target_max_dd_pct': 20.0,
 'target_relative_return_by_fold_pp': {'wf_2020': -263.7287958891489,
  'wf_2022': 30.0,
  'wf_2023_2024': 30.0},
 'wf2020_reference_floor_return_p

## Conclusion

The final cell writes `summary.json`, `selection.json`, `repair_plan.json`, and `conclusion.md`. The notebook remains the primary executable record for this diagnostic.
